In [1]:
from ultralytics import YOLO
import shutil
import os
import gc
import torch

DATA_YAML = "merged_dataset/data.yaml"

REPORT_DIR = "report_outputs"
os.makedirs(REPORT_DIR, exist_ok=True)

COMMON_ARGS = dict(
    data=DATA_YAML,
    epochs=200,
    imgsz=640,

    optimizer="AdamW",
    lr0=1e-4,
    weight_decay=1e-4,
    cos_lr=True,

    # Roboflow already augmented/preprocessed
    augment=False,

    # safer for RTX 4050 6GB
    amp=False,
    workers=4,

    project="runs",
    save=True,
    save_period=20,
    patience=50,
    device=0
)

def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def copy_report_files(run_dir, model_name):
    useful_files = [
        "results.png",
        "confusion_matrix.png",
        "confusion_matrix_normalized.png",
        "F1_curve.png",
        "P_curve.png",
        "R_curve.png",
        "PR_curve.png",
        "labels.jpg",
        "labels_correlogram.jpg"
    ]

    dst_dir = os.path.join(REPORT_DIR, model_name)
    os.makedirs(dst_dir, exist_ok=True)

    for file in useful_files:
        src = os.path.join(run_dir, file)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(dst_dir, file))

    weights_dir = os.path.join(run_dir, "weights")

    if os.path.exists(weights_dir):
        for weight_file in ["best.pt", "last.pt"]:
            src = os.path.join(weights_dir, weight_file)
            if os.path.exists(src):
                shutil.copy2(src, os.path.join(dst_dir, weight_file))

    print(f"Saved report files for {model_name}")

def train_model(model_file, run_name, batch_size):
    cleanup_gpu()

    print("\n==============================")
    print(f"TRAINING {run_name}")
    print("==============================")

    model = YOLO(model_file)

    results = model.train(
        **COMMON_ARGS,
        batch=batch_size,
        name=run_name
    )

    run_dir = str(results.save_dir)
    copy_report_files(run_dir, run_name)

    del model
    cleanup_gpu()

# Train sequentially
train_model("yolo11m.pt", "yolo11m_merged", 8)
train_model("yolo11l.pt", "yolo11l_merged", 4)
train_model("yolo11x.pt", "yolo11x_merged", 2)

print("\nALL TRAINING COMPLETE")
print(f"Report outputs saved to: {REPORT_DIR}")


TRAINING yolo11m_merged
New https://pypi.org/project/ultralytics/8.4.51 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.50  Python-3.11.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=merged_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.

KeyboardInterrupt: 

In [3]:
from ultralytics import YOLO
import os
import shutil

DATA_YAML = "merged_dataset_cleaned/data.yaml"
REPORT_DIR = "report_outputs"
os.makedirs(REPORT_DIR, exist_ok=True)

def copy_report_files(run_dir, model_name):
    files = [
        "results.png",
        "results.csv",
        "confusion_matrix.png",
        "confusion_matrix_normalized.png",
        "F1_curve.png",
        "P_curve.png",
        "R_curve.png",
        "PR_curve.png",
        "labels.jpg",
        "labels_correlogram.jpg",
    ]

    dst_dir = os.path.join(REPORT_DIR, model_name)
    os.makedirs(dst_dir, exist_ok=True)

    for f in files:
        src = os.path.join(run_dir, f)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(dst_dir, f))

    weights_dir = os.path.join(run_dir, "weights")
    if os.path.exists(weights_dir):
        for f in ["best.pt", "last.pt"]:
            src = os.path.join(weights_dir, f)
            if os.path.exists(src):
                shutil.copy2(src, os.path.join(dst_dir, f))

    print(f"Saved report outputs for {model_name} to {dst_dir}")

def train_and_save(model_file, name, args):
    model = YOLO(model_file)

    results = model.train(
        **args,
        name=name
    )

    copy_report_files(str(results.save_dir), name)

COMMON_ARGS = dict(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=8,
    optimizer="AdamW",
    lr0=1e-4,
    weight_decay=1e-4,
    cos_lr=True,
    augment=False,
    amp=True,
    workers=4,
    project="runs",
    save=True,
    save_period=5,
    patience=5,
    device=0
)

COMMON_ARGS2 = dict(
    data=DATA_YAML,
    epochs=10,
    imgsz=512,
    batch=8,
    optimizer="AdamW",
    lr0=1e-4,
    weight_decay=1e-4,
    cos_lr=True,
    augment=False,
    amp=True,
    workers=4,
    project="runs",
    save=True,
    save_period=5,
    patience=5,
    device=0
)

train_and_save("yolo11s.pt", "yolo11s_fast_640", COMMON_ARGS)
# train_and_save("yolo11m.pt", "yolo11m_fast_512", COMMON_ARGS2)

KeyboardInterrupt: 